[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-08-authentication-jwt.ipynb#scrollTo=11223344)

---
# Day 8 · Authentication — OAuth2, Password Hashing, and JWT
**certified-journeys / fastapi-certified** · Security Layer

> **Goal for today:** Implement a complete OAuth2 password-flow authentication system: hash passwords with passlib, issue JWT access tokens, protect routes with a `get_current_user` dependency, and verify that expired and invalid tokens are rejected.


In [ ]:
%pip install -q fastapi httpx "python-jose[cryptography]" "passlib[bcrypt]"


---
## The OAuth2 password flow — big picture

FastAPI's built-in `OAuth2PasswordBearer` implements the simplest OAuth2 grant type:

```
Client              FastAPI
  │                    │
  │──POST /token──────►│  username + password (form data)
  │                    │  verify password hash → issue JWT
  │◄──{access_token}───│
  │                    │
  │──GET /me ─────────►│  Authorization: Bearer <token>
  │                    │  decode JWT → get user
  │◄──{user data}──────│
```

Key components:

| Component | Library | Role |
|---|---|---|
| Password hashing | `passlib` | Store hashes, never plaintext |
| JWT encode/decode | `python-jose` | Stateless token, no DB lookup per request |
| Token endpoint | FastAPI + `OAuth2PasswordRequestForm` | Issues the JWT |
| Auth dependency | `OAuth2PasswordBearer` + custom dep | Extracts and validates JWT per route |


---
## Step 1 · Password hashing with passlib

**Never store plaintext passwords.** `passlib` wraps bcrypt (and other algorithms) with a clean API.

| Method | What it does |
|---|---|
| `CryptContext(schemes=['bcrypt'])` | Configures the hashing algorithm |
| `pwd_context.hash(plain)` | Returns a bcrypt hash string |
| `pwd_context.verify(plain, hashed)` | Constant-time comparison → `True`/`False` |

bcrypt hashes include a **random salt** — hashing the same password twice produces different strings. `verify()` knows how to extract the salt and compare correctly.


In [ ]:
from passlib.context import CryptContext

# schemes=['bcrypt'] — bcrypt is the recommended algorithm for passwords
# deprecated='auto'  — automatically marks old algorithms as deprecated
pwd_context = CryptContext(schemes=['bcrypt'], deprecated='auto')

plain_password = 'supersecret123'

# Hash the password — bcrypt includes a random salt, so each call produces a different string
hashed_1 = pwd_context.hash(plain_password)
hashed_2 = pwd_context.hash(plain_password)

print('Hash 1:', hashed_1[:30], '...')
print('Hash 2:', hashed_2[:30], '...')
print('Same hash?', hashed_1 == hashed_2)  # False — different salts

# Verify uses constant-time comparison to prevent timing attacks
print('Correct password?', pwd_context.verify(plain_password, hashed_1))  # True
print('Wrong password?  ', pwd_context.verify('wrongpass', hashed_1))     # False


**What just happened?**
- Each call to `pwd_context.hash()` embeds a **fresh random salt** — two hashes of the same password are always different.
- `pwd_context.verify()` extracts the salt from the stored hash and applies the same hash function to the plaintext — only the values match, not the strings.
- Passlib uses **constant-time comparison** to prevent timing attacks where an attacker infers correctness from response time.


---
## Step 2 · JWT configuration and token creation

A JWT (JSON Web Token) has three parts: **header** (algorithm), **payload** (claims like `sub`, `exp`), and **signature** (verifies the token hasn't been tampered with).

| Claim | Meaning |
|---|---|
| `sub` | Subject — typically the username or user ID |
| `exp` | Expiry timestamp — token rejected after this |
| `iat` | Issued-at timestamp (optional but useful) |

The `SECRET_KEY` must be a **long random string** in production. Generate one with `openssl rand -hex 32`.


In [ ]:
from datetime import datetime, timedelta, timezone
from jose import jwt, JWTError

# NEVER use this string in production — generate with: openssl rand -hex 32
SECRET_KEY = 'notebook-test-secret-key-do-not-use-in-production'
ALGORITHM  = 'HS256'                # HMAC-SHA256 — fast, symmetric
ACCESS_TOKEN_EXPIRE_MINUTES = 30    # 30-minute expiry is a common default


def create_access_token(data: dict, expires_delta: timedelta | None = None) -> str:
    """
    Encode a JWT with the given payload data plus an expiry claim.
    Returns the signed JWT string.
    """
    to_encode = data.copy()
    expire = datetime.now(timezone.utc) + (
        expires_delta if expires_delta else timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    )
    to_encode.update({'exp': expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)


# Create a token for user 'alice'
token = create_access_token(data={'sub': 'alice'})
print('JWT token (first 40 chars):', token[:40], '...')

# Decode to inspect payload (this also verifies the signature)
payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
print('Decoded payload:', payload)
print('Token expires at:', datetime.fromtimestamp(payload['exp'], tz=timezone.utc))


**What just happened?**
- `jwt.encode()` creates a signed token — the signature uses the `SECRET_KEY` so the server can verify it wasn't forged.
- `jwt.decode()` simultaneously **verifies the signature** and **checks the expiry** — a single call, no DB lookup required.
- The payload (`sub`, `exp`) is **not encrypted** — don't put secrets in JWT claims. It is only signed (tamper-proof).


---
## Step 3 · In-memory user store and authentication logic

For this notebook we keep a simple dict of users (in production, this would be your DB). The authenticate function:
1. Looks up the user by username
2. Verifies the password hash
3. Returns the user if valid, `None` if not


In [ ]:
from typing import Optional
from pydantic import BaseModel

# Pydantic models
class UserInDB(BaseModel):
    username: str
    email: str
    hashed_password: str
    disabled: bool = False

class UserPublic(BaseModel):
    username: str
    email: str
    disabled: bool


# In-memory user store — simulates a DB table
# Passwords are stored as bcrypt hashes (NEVER plaintext)
FAKE_USERS_DB: dict[str, UserInDB] = {
    'alice': UserInDB(
        username='alice',
        email='alice@example.com',
        hashed_password=pwd_context.hash('alicepassword'),
    ),
    'bob': UserInDB(
        username='bob',
        email='bob@example.com',
        hashed_password=pwd_context.hash('bobpassword'),
        disabled=True,   # inactive account
    ),
}


def get_user(username: str) -> Optional[UserInDB]:
    return FAKE_USERS_DB.get(username)


def authenticate_user(username: str, password: str) -> Optional[UserInDB]:
    """Verify credentials. Returns user if valid, None otherwise."""
    user = get_user(username)
    if user is None:
        return None
    if not pwd_context.verify(password, user.hashed_password):
        return None
    return user


# Test the authentication logic
print('Valid credentials:', authenticate_user('alice', 'alicepassword').username)
print('Wrong password:   ', authenticate_user('alice', 'wrongpass'))
print('Unknown user:     ', authenticate_user('eve', 'anything'))


**What just happened?**
- `authenticate_user` never reveals *why* authentication failed — always return `None` (not "bad password" vs "user not found") to prevent user enumeration attacks.
- Hashed passwords are stored at startup; `pwd_context.hash()` is called once, not on every request.
- The `disabled` flag models inactive accounts — you'll use it in the `get_current_user` dependency.


---
## Step 4 · The `/token` endpoint

The token endpoint accepts `application/x-www-form-urlencoded` (OAuth2 standard), not JSON. FastAPI provides `OAuth2PasswordRequestForm` for this.

The endpoint:
1. Reads `username` and `password` from form data
2. Calls `authenticate_user`
3. Issues a JWT if credentials are valid
4. Returns `{"access_token": "...", "token_type": "bearer"}`


In [ ]:
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm

app = FastAPI(title='FastAPI JWT Auth Demo')

# tokenUrl tells Swagger UI where to send login requests
# It also declares the scheme used by the security dependency below
oauth2_scheme = OAuth2PasswordBearer(tokenUrl='token')


class Token(BaseModel):
    access_token: str
    token_type: str


@app.post('/token', response_model=Token)
def login_for_access_token(form_data: OAuth2PasswordRequestForm = Depends()):
    """
    OAuth2 password flow token endpoint.
    Accepts form data (not JSON) per the OAuth2 spec.
    """
    user = authenticate_user(form_data.username, form_data.password)
    if not user:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail='Incorrect username or password',
            # WWW-Authenticate header is required by the OAuth2 spec
            headers={'WWW-Authenticate': 'Bearer'},
        )
    access_token = create_access_token(
        data={'sub': user.username},
        expires_delta=timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES),
    )
    return Token(access_token=access_token, token_type='bearer')


print('App created with /token endpoint.')


**What just happened?**
- `OAuth2PasswordRequestForm` reads `username` and `password` from a **form-encoded body** (not JSON). This is the OAuth2 spec requirement.
- The `WWW-Authenticate: Bearer` header in the 401 response tells the client which authentication scheme to use.
- `oauth2_scheme = OAuth2PasswordBearer(tokenUrl='token')` is a **dependency** that extracts the Bearer token from the `Authorization` header on protected routes.


---
## Step 5 · `get_current_user` dependency

This dependency sits between the JWT token and the route. It:
1. Receives the raw Bearer token from `oauth2_scheme`
2. Decodes and validates the JWT
3. Looks up the user from the `sub` claim
4. Raises `401` for any failure (invalid, expired, missing)

A second dependency `get_current_active_user` adds a **disabled-user check** on top.


In [ ]:
CREDENTIALS_EXCEPTION = HTTPException(
    status_code=status.HTTP_401_UNAUTHORIZED,
    detail='Could not validate credentials',
    headers={'WWW-Authenticate': 'Bearer'},
)


def get_current_user(token: str = Depends(oauth2_scheme)) -> UserInDB:
    """
    Dependency: decodes the JWT Bearer token and returns the authenticated user.
    Raises 401 if the token is invalid, expired, or the user does not exist.
    """
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username: str = payload.get('sub')   # the subject claim holds the username
        if username is None:
            raise CREDENTIALS_EXCEPTION
    except JWTError:
        # JWTError covers: invalid signature, expired token, malformed token
        raise CREDENTIALS_EXCEPTION

    user = get_user(username)
    if user is None:
        raise CREDENTIALS_EXCEPTION
    return user


def get_current_active_user(
    current_user: UserInDB = Depends(get_current_user),
) -> UserInDB:
    """Dependency: additionally rejects disabled accounts."""
    if current_user.disabled:
        raise HTTPException(status_code=400, detail='Inactive user')
    return current_user


# ── Protected route ───────────────────────────────────────────────────────────
@app.get('/users/me', response_model=UserPublic)
def read_users_me(current_user: UserInDB = Depends(get_current_active_user)):
    """Returns the currently authenticated user's profile."""
    return current_user


@app.get('/items/me')
def read_own_items(current_user: UserInDB = Depends(get_current_active_user)):
    """Example of a data endpoint gated behind auth."""
    return [{'item': 'Foo', 'owner': current_user.username}]


print('Protected routes registered.')


**What just happened?**
- `get_current_user` **catches all JWT failure modes** with a single `except JWTError` — expired, tampered, and malformed tokens all raise the same 401.
- The chain `get_current_active_user → get_current_user → oauth2_scheme` is a **dependency chain** — FastAPI builds and executes it automatically.
- Separating `get_current_user` (validates token) from `get_current_active_user` (checks active status) lets you reuse either independently.


---
## Step 6 · Testing the full auth flow with TestClient

We test the complete token issuance and protected route access cycle using `TestClient`. The token is obtained via a `POST /token` form request, then attached as a `Bearer` header on subsequent requests.


In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# ── 1. Get a token with valid credentials ──────────────────────────────────────
resp = client.post('/token', data={'username': 'alice', 'password': 'alicepassword'})
assert resp.status_code == 200, resp.text
token_data = resp.json()
print('Token response:', {k: v[:20] + '...' if k == 'access_token' else v for k, v in token_data.items()})

token = token_data['access_token']
headers = {'Authorization': f'Bearer {token}'}

# ── 2. Access a protected route with the token ─────────────────────────────────
resp = client.get('/users/me', headers=headers)
assert resp.status_code == 200
print('Authenticated user:', resp.json())

# ── 3. Reject request with no token ───────────────────────────────────────────
resp = client.get('/users/me')  # no Authorization header
assert resp.status_code == 401
print('No token → status:', resp.status_code)

# ── 4. Reject request with wrong credentials at /token ────────────────────────
resp = client.post('/token', data={'username': 'alice', 'password': 'wrongpass'})
assert resp.status_code == 401
print('Wrong password → status:', resp.status_code, '|', resp.json()['detail'])

# ── 5. Reject disabled user (bob) ─────────────────────────────────────────────
resp = client.post('/token', data={'username': 'bob', 'password': 'bobpassword'})
assert resp.status_code == 200  # token IS issued for disabled users
bob_token = resp.json()['access_token']
resp = client.get('/users/me', headers={'Authorization': f'Bearer {bob_token}'})
assert resp.status_code == 400   # but /users/me rejects disabled users
print('Disabled user → status:', resp.status_code, '|', resp.json()['detail'])


**What just happened?**
- `POST /token` with `data=` sends a **form-encoded** body (not JSON) — that's the OAuth2 spec.
- The `Authorization: Bearer <token>` header is how the client proves identity on every subsequent request.
- Disabled user `bob` can **get a token** (authentication succeeded) but **not access protected routes** (authorization failed) — a deliberate two-step design.


---
## Step 7 · Token expiry — verify that expired tokens are rejected

JWT expiry is enforced by `python-jose` during `jwt.decode()`. We test this by manually creating a token that expired in the past.


In [ ]:
from datetime import timezone

# Create a token that expired 1 minute ago
expired_token = create_access_token(
    data={'sub': 'alice'},
    expires_delta=timedelta(minutes=-1),  # negative delta → already expired!
)

# Inspect the expiry claim before testing
payload = jwt.decode(expired_token, SECRET_KEY, algorithms=[ALGORITHM], options={'verify_exp': False})
exp_dt = datetime.fromtimestamp(payload['exp'], tz=timezone.utc)
print('Expired token exp:', exp_dt, '(in the past)')

# Attempt to use the expired token on a protected route
resp = client.get('/users/me', headers={'Authorization': f'Bearer {expired_token}'})
assert resp.status_code == 401, resp.text
print('Expired token → status:', resp.status_code, '|', resp.json()['detail'])

# Also verify a completely tampered token is rejected
tampered = expired_token[:-5] + 'XXXXX'   # corrupt the signature
resp = client.get('/users/me', headers={'Authorization': f'Bearer {tampered}'})
assert resp.status_code == 401
print('Tampered token → status:', resp.status_code, '|', resp.json()['detail'])

# And a token signed with a different secret is rejected
wrong_key_token = jwt.encode({'sub': 'alice', 'exp': datetime.now(timezone.utc) + timedelta(minutes=30)}, 'wrong-secret', algorithm='HS256')
resp = client.get('/users/me', headers={'Authorization': f'Bearer {wrong_key_token}'})
assert resp.status_code == 401
print('Wrong key token → status:', resp.status_code, '|', resp.json()['detail'])


**What just happened?**
- `timedelta(minutes=-1)` creates a token with `exp` in the past — `jwt.decode()` raises `JWTError` automatically.
- **Three rejection paths** all produce `401`: expired token, tampered signature, wrong signing key.
- `options={'verify_exp': False}` in the inspection-only decode lets us read the payload without triggering the expiry check — useful for debugging.


---
## Step 8 · Scopes — fine-grained authorization (overview)

For larger applications, a single `is_authenticated` check isn't enough. OAuth2 **scopes** let you declare what a token is allowed to do.

FastAPI provides `SecurityScopes` for scope-based authorization:

```python
from fastapi.security import SecurityScopes

# Declare scopes when creating the scheme
oauth2_scheme = OAuth2PasswordBearer(
    tokenUrl='token',
    scopes={'items:read': 'Read items', 'items:write': 'Write items'},
)

# Require scopes on a route
@app.get('/items', dependencies=[Security(get_current_user, scopes=['items:read'])])
def list_items(): ...
```

The scopes are embedded in the JWT payload and verified by the dependency.
For this course we keep it simple — the pattern above is the production extension.


In [ ]:
# Demonstrate embedding scopes in a token and reading them back
scoped_token = create_access_token(data={'sub': 'alice', 'scopes': ['items:read', 'items:write']})

payload = jwt.decode(scoped_token, SECRET_KEY, algorithms=[ALGORITHM])
print('Token subject:', payload.get('sub'))
print('Token scopes: ', payload.get('scopes'))

# A dependency would check: if required_scope not in payload.get('scopes', []):
#     raise HTTPException(403, 'Insufficient scope')
required_scope = 'items:read'
has_scope = required_scope in payload.get('scopes', [])
print(f'Has scope {required_scope!r}:', has_scope)


**What just happened?**
- Scopes are just **extra claims** in the JWT payload — a list of strings like `['items:read', 'items:write']`.
- The dependency reads the scopes from the decoded token and compares them against what the route requires.
- Use `403 Forbidden` (not 401) when the token is valid but lacks the required scope — authenticated but not authorized.


---
## Step 9 · Refresh tokens — the concept

Short-lived access tokens (15–60 min) minimize the damage if a token is stolen. **Refresh tokens** let the client get a new access token without re-entering credentials.

| Token type | Lifetime | Stored |
|---|---|---|
| Access token | 15–60 min | Memory (not `localStorage`) |
| Refresh token | Days–weeks | `httpOnly` cookie or secure DB |

Flow:
```
POST /token        → {access_token, refresh_token}
... access token expires ...
POST /token/refresh → {new_access_token}
```

Implementation pattern: store the refresh token's `jti` (JWT ID) in the database and invalidate it on logout — this is how you implement "log out all devices".


In [ ]:
# Challenge: Add a /users/me/items endpoint that:
# 1. Requires authentication (Depends(get_current_active_user))
# 2. Returns a list of fake items for the current user
# 3. Test it with TestClient using a valid token
# 4. Bonus: verify it returns 401 with no token

# Hint: the route body should look like:
# return [{'title': 'My item', 'owner': current_user.username}]

# Your solution here:

# @app.get('/users/me/items')
# def read_my_items(current_user: UserInDB = Depends(get_current_active_user)):
#     pass

# Then test it:
# resp = client.post('/token', data={'username': 'alice', 'password': 'alicepassword'})
# token = resp.json()['access_token']
# ...


---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| `CryptContext(schemes=['bcrypt'])` | Hash with `pwd_context.hash()`, verify with `pwd_context.verify()` |
| `jwt.encode(payload, key, alg)` | Signs the payload — not encrypted, only tamper-proof |
| `jwt.decode(token, key, algs)` | Verifies signature AND expiry — raises `JWTError` on failure |
| `OAuth2PasswordRequestForm` | Reads `username`+`password` from form-encoded body (not JSON) |
| `OAuth2PasswordBearer(tokenUrl=...)` | Dependency that extracts Bearer token from `Authorization` header |
| `get_current_user` dependency | Decodes JWT, looks up user — single source of auth truth |
| `WWW-Authenticate: Bearer` header | Required by OAuth2 spec on 401 responses |
| `JWTError` catch-all | Covers expired, tampered, and malformed tokens |

> **Tip:** Store only the hashed password — never the plaintext. Use passlib's CryptContext with `schemes=['bcrypt']`.

---
## What's next
**Day 9** → Background tasks (fire-and-forget), custom request-timing middleware, CORS configuration, and GZip compression.

Mark Day 8 complete in your [tracker](../index.html).
